# 🎯 Project 6: Sentiment Analysis on Social Media Data

**Objective:** Perform sentiment analysis on social media-style text data to classify sentiments as **Positive**, **Negative**, or **Neutral**.

**Skills Covered:**
- Natural Language Processing (NLP)
- Sentiment Analysis using `TextBlob` and `VADER (NLTK)`
- Data Cleaning & Preprocessing
- Visualization of Results

## 📦 Step 1: Install Required Libraries

In [ ]:
# Install required packages
!pip install textblob vaderSentiment wordcloud -q

import nltk
nltk.download('vader_lexicon')
nltk.download('punkt')
nltk.download('stopwords')

print("✅ All libraries installed and downloaded!")

## 📚 Step 2: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import re
import string
from collections import Counter

# NLP Libraries
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from wordcloud import WordCloud

# Styling
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')

print("✅ All libraries imported successfully!")

## 📊 Step 3: Load / Create Social Media Dataset

> ℹ️ **Note:** Since direct Twitter API access requires credentials, we use a realistic simulated dataset of 100 tweets covering diverse topics (tech, food, politics, sports, etc.). You can replace this with your own CSV file.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Simulated Social Media Dataset (100 realistic tweets)
# ─────────────────────────────────────────────────────────────

tweets_data = [
    # POSITIVE
    ("I absolutely love the new iPhone! Best purchase ever! 😍 #Apple #Tech", "Technology"),
    ("Just got promoted at work!! So thrilled and grateful for this opportunity 🎉", "Personal"),
    ("The weather today is absolutely gorgeous! Perfect day for a picnic ☀️", "Weather"),
    ("This movie was a masterpiece. One of the best films I've seen in years!", "Entertainment"),
    ("Amazing meal at this new restaurant. The pasta was divine! ⭐⭐⭐⭐⭐", "Food"),
    ("Just finished my first 5K run! So proud of myself! 🏃‍♂️💪 #Fitness", "Sports"),
    ("Python is such a beautiful programming language. Love coding! #Python", "Technology"),
    ("My team won the championship today!!! Can't stop smiling 😊⚽ #Football", "Sports"),
    ("Incredible concert last night! The energy was phenomenal! 🎵🎶", "Entertainment"),
    ("Finally got my dream job! Years of hard work paid off! 🙌 #Grateful", "Personal"),
    ("The new update is fantastic! So many great features added 👍", "Technology"),
    ("Spent the day volunteering at the shelter. So rewarding! ❤️", "Personal"),
    ("Best vacation ever! The beaches were stunning and food was amazing 🏖️", "Travel"),
    ("Loving the new season of my favorite show! So well written!", "Entertainment"),
    ("Great customer service experience! They went above and beyond! ⭐", "Business"),
    ("Just adopted a puppy! Life is so much better with dogs 🐶❤️", "Personal"),
    ("The renewable energy sector is growing faster than ever! Great news 🌱", "Environment"),
    ("Had a wonderful family dinner tonight. Cherishing these moments 💛", "Personal"),
    ("New AI tools are transforming productivity! Exciting times ahead 🚀", "Technology"),
    ("The sunrise this morning was absolutely breathtaking 🌅 #Nature", "Nature"),
    ("So happy with my new laptop! Super fast and the display is gorgeous", "Technology"),
    ("Thank you to everyone who supported me through this journey! 🙏", "Personal"),
    ("Yoga retreat was transformative! Feeling so peaceful and centered ✨", "Health"),
    ("The new coffee shop downtown is amazing! Best lattes in the city ☕", "Food"),
    ("Our local park was cleaned up today by volunteers. Community rocks! 🌳", "Community"),

    # NEGATIVE
    ("Worst customer service I've ever experienced. Never going back! 😡", "Business"),
    ("This phone died after just 6 months. Complete waste of money! 📱💔", "Technology"),
    ("Traffic was absolutely terrible this morning. 2 hours stuck! 😤", "Transport"),
    ("The movie was so boring and predictable. Total disappointment 😒", "Entertainment"),
    ("Terrible weather, rained all day, cancelled all plans. So frustrated!", "Weather"),
    ("Lost my job today. Not sure what I'm going to do now 😢 #Unemployed", "Personal"),
    ("Restaurant was disgusting. Found hair in my food and the service was slow 🤢", "Food"),
    ("Can't believe how expensive everything has become. This inflation is killing us!", "Economy"),
    ("Got scammed online. Lost so much money. People are just cruel 😭", "Personal"),
    ("Our team lost again. This season has been a complete disaster 😞⚽", "Sports"),
    ("The new update broke everything! My app doesn't work anymore! 😠 #Bug", "Technology"),
    ("Terrible hospital experience. Waited 6 hours and still wasn't seen 😤", "Health"),
    ("Climate change is destroying our planet and nobody seems to care 🌍😢", "Environment"),
    ("Horrible flight delay, missed my connection, trip ruined. Never fly this airline again!", "Travel"),
    ("Can't sleep again. Insomnia is ruining my health and my work 😩", "Health"),
    ("So disappointed with this election result. This is not the future we wanted", "Politics"),
    ("My new headphones broke on day 2. Completely defective product 👎", "Technology"),
    ("Rent went up by 30%. I can't afford to live here anymore 😭 #HousingCrisis", "Economy"),
    ("The concert was poorly organized. Terrible sound quality and overcrowded 😤", "Entertainment"),
    ("Getting really tired of all the fake news spreading everywhere 😡 #Media", "Politics"),
    ("Failed my exam after studying all night. Feel so defeated 😢", "Education"),
    ("This new policy is going to hurt so many working families. Outrageous!", "Politics"),
    ("Worst road trip ever. Car broke down, no signal, and it rained the whole time ⛈️", "Travel"),
    ("Feeling so lonely lately. Social media just makes it worse sometimes 😔", "Personal"),
    ("The noise pollution in this city is unbearable. Can't concentrate on anything", "Environment"),

    # NEUTRAL
    ("Just read an article about climate change. There are multiple perspectives to consider.", "Environment"),
    ("The government announced new tax policies today. Details still being analyzed.", "Politics"),
    ("Apple released iOS 18 today. Here are the new features included in the update.", "Technology"),
    ("Watched the game last night. Both teams played as expected.", "Sports"),
    ("The new café opened on 5th street. It serves coffee and sandwiches.", "Food"),
    ("Today is Monday. The week has officially begun.", "General"),
    ("GDP growth was reported at 2.1% for the last quarter by the finance ministry.", "Economy"),
    ("The meeting has been rescheduled to Thursday at 3 PM.", "Business"),
    ("Scientists discovered a new species of fish in the Pacific Ocean.", "Science"),
    ("The library will be closed on public holidays as per the updated schedule.", "General"),
    ("New study suggests moderate exercise has health benefits. Participants were surveyed over 6 months.", "Health"),
    ("The train from Delhi to Mumbai departs at 6:00 AM and arrives at 8:00 PM.", "Transport"),
    ("Local elections are scheduled for next month. Voter registration is now open.", "Politics"),
    ("The company reported quarterly earnings in line with analyst expectations.", "Business"),
    ("A new museum exhibit opens this weekend showcasing ancient Roman artifacts.", "Culture"),
    ("Weather forecast predicts partly cloudy skies with a chance of rain tomorrow.", "Weather"),
    ("The semester begins on August 1st. Course enrollment is now available online.", "Education"),
    ("A new study on social media usage patterns has been published in the journal.", "Research"),
    ("The city council voted 5-4 to approve the new budget for the fiscal year.", "Politics"),
    ("Netflix added 30 new titles to its library this month across various genres.", "Entertainment"),
    ("The construction on Main Street is expected to finish by December.", "General"),
    ("Researchers are studying the effects of diet on long-term cognitive health.", "Science"),
    ("The airline has updated its baggage policy effective from next month.", "Travel"),
    ("The summit between world leaders concluded after three days of discussions.", "Politics"),
    ("Population in urban areas has grown by 15% over the past decade, per census data.", "Research"),

    # MIXED / SARCASTIC (challenging cases)
    ("Oh great, another software update that slows everything down. Just what I needed 🙄", "Technology"),
    ("Sure, because raising prices again is EXACTLY what customers wanted... 🙄 #Sarcasm", "Business"),
    ("Amazing how the WiFi works perfectly everywhere EXCEPT at home 🙄", "Technology"),
    ("Wonderful! My delivery arrived a week late and broken. Best service ever! 🙄", "Business"),
    ("Had a rollercoaster of a day. Some wins, some losses. Life goes on I guess.", "Personal"),
    ("The food was okay, nothing special. Expected more given the hype and the price.", "Food"),
    ("Mixed feelings about the new policy. Some good points but also real concerns.", "Politics"),
    ("The movie had great visuals but the story was weak. Average overall.", "Entertainment"),
    ("Tried the new workout app. It's decent but missing some key features I need.", "Health"),
    ("The city has improved but still has a long way to go on public transport.", "Transport"),
    ("Weekend was a mix of relaxing and stressful. Hard to say if it was good.", "Personal"),
    ("The product is fine. Not amazing, not terrible. Does the job I suppose.", "Business"),
    ("Slightly better than expected but still not worth the price they're charging.", "Business"),
    ("So the 'smart' speaker couldn't understand my accent again. Groundbreaking AI. 🙄", "Technology"),
    ("Can't decide if I love or hate working from home. Both at the same time maybe?", "Personal"),
    ("Yeah sure, the traffic 'improved' — if you call 45 minutes better than 1 hour improved.", "Transport"),
    ("The hotel was nice but the pool was closed and breakfast was disappointing.", "Travel"),
    ("Glad the event happened but the organization was frankly chaotic.", "Entertainment"),
    ("My boss thanked the team… but still didn't approve the raises we asked for. Progress?", "Business"),
    ("The app updated and now has more ads. Thanks for the 'improvement'! 🙄", "Technology"),
    ("New year, same problems. At least the coffee is still good ☕", "Personal"),
    ("Phone battery went from bad to slightly less bad with the update. Growth I guess.", "Technology"),
    ("Survived another Monday. That's something, right? 😅", "Personal"),
    ("The presentation went okay. Not a disaster, not a success. Just... okay.", "Business"),
    ("Well, at least the rain stopped. Small mercies I suppose 🌤️", "Weather"),
]

# Create DataFrame
df = pd.DataFrame(tweets_data, columns=['text', 'category'])
df['tweet_id'] = range(1, len(df) + 1)
df['username'] = [f"user_{np.random.randint(1000, 9999)}" for _ in range(len(df))]
df['likes'] = np.random.randint(0, 5000, len(df))
df['retweets'] = np.random.randint(0, 1000, len(df))

print(f"✅ Dataset created with {len(df)} tweets!")
print(f"\n📂 Categories: {df['category'].unique()}")
df.head(10)

## 🧹 Step 4: Data Cleaning & Preprocessing

In [ ]:
def clean_tweet(text):
    """
    Cleans a tweet by:
    - Removing URLs
    - Removing @mentions
    - Removing hashtag symbols (keeping the word)
    - Removing emojis and special characters
    - Converting to lowercase
    - Removing extra whitespace
    """
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)   # Remove URLs
    text = re.sub(r'@\w+', '', text)                       # Remove @mentions
    text = re.sub(r'#(\w+)', r'\1', text)                  # Remove # but keep word
    text = re.sub(r'[^\x00-\x7F]+', '', text)              # Remove non-ASCII (emojis)
    text = text.translate(str.maketrans('', '', string.punctuation))  # Remove punctuation
    text = text.lower()                                     # Lowercase
    text = re.sub(r'\s+', ' ', text).strip()               # Remove extra whitespace
    return text

def remove_stopwords(text):
    """Removes common stopwords from text."""
    stop_words = set(stopwords.words('english'))
    tokens = word_tokenize(text)
    filtered = [w for w in tokens if w not in stop_words and len(w) > 2]
    return ' '.join(filtered)

# Apply cleaning
df['cleaned_text'] = df['text'].apply(clean_tweet)
df['processed_text'] = df['cleaned_text'].apply(remove_stopwords)

# Check for duplicates and missing values
print("🔍 Data Quality Check:")
print(f"   Total Tweets: {len(df)}")
print(f"   Missing Values: {df.isnull().sum().sum()}")
print(f"   Duplicate Tweets: {df.duplicated(subset='text').sum()}")
print("\n📋 Sample Cleaned Data:")
df[['text', 'cleaned_text', 'processed_text']].head(5)

## 🤖 Step 5: Sentiment Analysis

We'll use **two approaches** and compare them:
1. **TextBlob** — Pattern-based, uses polarity & subjectivity scores
2. **VADER (NLTK)** — Designed for social media text, handles emojis, slang, and punctuation

In [ ]:
# ─────────────────────────────────────────────────────
# METHOD 1: TextBlob Sentiment Analysis
# ─────────────────────────────────────────────────────

def textblob_sentiment(text):
    analysis = TextBlob(text)
    polarity = analysis.sentiment.polarity      # -1 (negative) to +1 (positive)
    subjectivity = analysis.sentiment.subjectivity  # 0 (objective) to 1 (subjective)

    if polarity > 0.05:
        label = 'Positive'
    elif polarity < -0.05:
        label = 'Negative'
    else:
        label = 'Neutral'

    return pd.Series([polarity, subjectivity, label])

df[['tb_polarity', 'tb_subjectivity', 'tb_sentiment']] = df['cleaned_text'].apply(textblob_sentiment)

print("✅ TextBlob Analysis Complete!")
print("\nTextBlob Sentiment Distribution:")
print(df['tb_sentiment'].value_counts())

In [ ]:
# ─────────────────────────────────────────────────────
# METHOD 2: VADER Sentiment Analysis
# ─────────────────────────────────────────────────────

analyzer = SentimentIntensityAnalyzer()

def vader_sentiment(text):
    scores = analyzer.polarity_scores(text)
    compound = scores['compound']

    if compound >= 0.05:
        label = 'Positive'
    elif compound <= -0.05:
        label = 'Negative'
    else:
        label = 'Neutral'

    return pd.Series([scores['pos'], scores['neg'], scores['neu'], compound, label])

# Apply VADER on original text (VADER handles emojis & punctuation better)
df[['vader_pos', 'vader_neg', 'vader_neu', 'vader_compound', 'vader_sentiment']] = df['text'].apply(vader_sentiment)

print("✅ VADER Analysis Complete!")
print("\nVADER Sentiment Distribution:")
print(df['vader_sentiment'].value_counts())

In [ ]:
# ─────────────────────────────────────────────────────
# View Full Results
# ─────────────────────────────────────────────────────

result_cols = ['text', 'category', 'tb_polarity', 'tb_sentiment', 'vader_compound', 'vader_sentiment']
print("📊 Sample Sentiment Analysis Results:")
df[result_cols].head(20)

## 📈 Step 6: Visualizations

In [ ]:
# ─────────────────────────────────────────────────────
# PLOT 1: Sentiment Distribution Comparison
# ─────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Sentiment Distribution Comparison', fontsize=16, fontweight='bold', y=1.02)

colors = {'Positive': '#2ecc71', 'Negative': '#e74c3c', 'Neutral': '#3498db'}

# TextBlob Pie Chart
tb_counts = df['tb_sentiment'].value_counts()
axes[0].pie(
    tb_counts.values,
    labels=tb_counts.index,
    colors=[colors[s] for s in tb_counts.index],
    autopct='%1.1f%%',
    startangle=90,
    explode=[0.05] * len(tb_counts),
    shadow=True
)
axes[0].set_title('TextBlob Sentiment', fontsize=13, fontweight='bold')

# VADER Pie Chart
vader_counts = df['vader_sentiment'].value_counts()
axes[1].pie(
    vader_counts.values,
    labels=vader_counts.index,
    colors=[colors[s] for s in vader_counts.index],
    autopct='%1.1f%%',
    startangle=90,
    explode=[0.05] * len(vader_counts),
    shadow=True
)
axes[1].set_title('VADER Sentiment', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('sentiment_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Plot 1 saved!")

In [ ]:
# ─────────────────────────────────────────────────────
# PLOT 2: Polarity Score Distribution (Histogram)
# ─────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Polarity / Compound Score Distributions', fontsize=15, fontweight='bold')

axes[0].hist(df['tb_polarity'], bins=20, color='#9b59b6', edgecolor='white', alpha=0.85)
axes[0].axvline(x=0.05, color='green', linestyle='--', label='Positive threshold')
axes[0].axvline(x=-0.05, color='red', linestyle='--', label='Negative threshold')
axes[0].set_title('TextBlob Polarity Scores', fontsize=12)
axes[0].set_xlabel('Polarity Score (-1 to +1)')
axes[0].set_ylabel('Number of Tweets')
axes[0].legend()

axes[1].hist(df['vader_compound'], bins=20, color='#e67e22', edgecolor='white', alpha=0.85)
axes[1].axvline(x=0.05, color='green', linestyle='--', label='Positive threshold')
axes[1].axvline(x=-0.05, color='red', linestyle='--', label='Negative threshold')
axes[1].set_title('VADER Compound Scores', fontsize=12)
axes[1].set_xlabel('Compound Score (-1 to +1)')
axes[1].set_ylabel('Number of Tweets')
axes[1].legend()

plt.tight_layout()
plt.savefig('polarity_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Plot 2 saved!")

In [ ]:
# ─────────────────────────────────────────────────────
# PLOT 3: Sentiment by Category (Stacked Bar)
# ─────────────────────────────────────────────────────

category_sentiment = df.groupby(['category', 'vader_sentiment']).size().unstack(fill_value=0)

# Keep top 10 categories by tweet count
top_cats = df['category'].value_counts().head(10).index
category_sentiment = category_sentiment.loc[category_sentiment.index.isin(top_cats)]

fig, ax = plt.subplots(figsize=(14, 6))

bar_colors = [colors.get(col, '#95a5a6') for col in category_sentiment.columns]
category_sentiment.plot(kind='bar', stacked=True, ax=ax, color=bar_colors, edgecolor='white', linewidth=0.5)

ax.set_title('Sentiment Distribution by Tweet Category (VADER)', fontsize=14, fontweight='bold')
ax.set_xlabel('Category', fontsize=11)
ax.set_ylabel('Number of Tweets', fontsize=11)
ax.legend(title='Sentiment', loc='upper right')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('sentiment_by_category.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Plot 3 saved!")

In [ ]:
# ─────────────────────────────────────────────────────
# PLOT 4: TextBlob vs VADER Agreement Heatmap
# ─────────────────────────────────────────────────────

confusion = pd.crosstab(
    df['tb_sentiment'],
    df['vader_sentiment'],
    rownames=['TextBlob'],
    colnames=['VADER']
)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    confusion, annot=True, fmt='d', cmap='Blues',
    linewidths=1, linecolor='white', ax=ax,
    cbar_kws={'label': 'Tweet Count'}
)
ax.set_title('TextBlob vs VADER Agreement Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('agreement_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Calculate agreement rate
agreement = (df['tb_sentiment'] == df['vader_sentiment']).sum()
print(f"\n📊 Model Agreement Rate: {agreement}/{len(df)} tweets ({agreement/len(df)*100:.1f}%)")

In [ ]:
# ─────────────────────────────────────────────────────
# PLOT 5: Word Clouds for Each Sentiment
# ─────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Word Clouds by Sentiment (VADER)', fontsize=15, fontweight='bold')

sentiment_colors = {
    'Positive': 'Greens',
    'Negative': 'Reds',
    'Neutral':  'Blues'
}

for ax, sentiment in zip(axes, ['Positive', 'Negative', 'Neutral']):
    subset = df[df['vader_sentiment'] == sentiment]['processed_text']
    text_corpus = ' '.join(subset.tolist())

    if text_corpus.strip():
        wc = WordCloud(
            width=500, height=350,
            background_color='white',
            colormap=sentiment_colors[sentiment],
            max_words=50,
            collocations=False
        ).generate(text_corpus)
        ax.imshow(wc, interpolation='bilinear')
    else:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', fontsize=14)

    ax.axis('off')
    ax.set_title(f'{sentiment} Tweets', fontsize=13, fontweight='bold',
                 color=list(colors.values())[list(colors.keys()).index(sentiment)])

plt.tight_layout()
plt.savefig('wordclouds.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Plot 5 saved!")

In [ ]:
# ─────────────────────────────────────────────────────
# PLOT 6: Polarity vs Subjectivity Scatter Plot
# ─────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(10, 6))

for sentiment, color in colors.items():
    subset = df[df['tb_sentiment'] == sentiment]
    ax.scatter(
        subset['tb_polarity'],
        subset['tb_subjectivity'],
        c=color, label=sentiment, alpha=0.7, s=80, edgecolors='white', linewidth=0.5
    )

ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5, linewidth=1)
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, linewidth=1)
ax.set_xlabel('Polarity (Negative → Positive)', fontsize=12)
ax.set_ylabel('Subjectivity (Objective → Subjective)', fontsize=12)
ax.set_title('Polarity vs Subjectivity (TextBlob)', fontsize=14, fontweight='bold')
ax.legend(title='Sentiment', fontsize=10)

# Annotate quadrants
ax.text(0.7, 0.9, 'Positive\n& Subjective', fontsize=8, color='#2ecc71', ha='center', transform=ax.transAxes)
ax.text(0.15, 0.9, 'Negative\n& Subjective', fontsize=8, color='#e74c3c', ha='center', transform=ax.transAxes)

plt.tight_layout()
plt.savefig('polarity_subjectivity_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Plot 6 saved!")

## 🔍 Step 7: Test with Your Own Tweets!

In [ ]:
def analyze_custom_text(text):
    """Analyze sentiment of any custom text using both methods."""
    print(f"\n📝 Text: {text}")
    print("-" * 60)

    # TextBlob
    cleaned = clean_tweet(text)
    tb = TextBlob(cleaned)
    tb_pol = tb.sentiment.polarity
    tb_sub = tb.sentiment.subjectivity
    tb_label = 'Positive' if tb_pol > 0.05 else ('Negative' if tb_pol < -0.05 else 'Neutral')

    # VADER
    scores = analyzer.polarity_scores(text)
    compound = scores['compound']
    vader_label = 'Positive' if compound >= 0.05 else ('Negative' if compound <= -0.05 else 'Neutral')

    emoji_map = {'Positive': '✅ 😊', 'Negative': '❌ 😠', 'Neutral': '⚪ 😐'}

    print(f"🔷 TextBlob  → Polarity: {tb_pol:+.3f} | Subjectivity: {tb_sub:.3f} | Sentiment: {emoji_map[tb_label]} {tb_label}")
    print(f"🔶 VADER     → Compound: {compound:+.3f} | Pos: {scores['pos']:.2f} | Neg: {scores['neg']:.2f} | Neu: {scores['neu']:.2f} | Sentiment: {emoji_map[vader_label]} {vader_label}")

    if tb_label == vader_label:
        print(f"\n🤝 Both models agree: {tb_label}")
    else:
        print(f"\n⚠️  Models disagree! TextBlob: {tb_label} | VADER: {vader_label}")

# ── Test Cases ──
test_tweets = [
    "I absolutely love this! Best day of my life! 😍🎉",
    "This is the worst experience I've ever had. Totally disgusting!",
    "The meeting is at 3 PM today.",
    "Oh great, another update that breaks everything. Thanks a lot! 🙄",
    "Feeling okay today, not great but not terrible either.",
    "Can't believe how amazing the sunset looks right now 🌅",
]

for tweet in test_tweets:
    analyze_custom_text(tweet)

In [ ]:
# ─────────────────────────────────────────────────────
# 💬 Analyze YOUR OWN text here!
# ─────────────────────────────────────────────────────

my_tweet = "I just can't believe how beautiful this day turned out to be! 🌟"   # ← CHANGE THIS

analyze_custom_text(my_tweet)

## 📋 Step 8: Summary & Key Insights

In [ ]:
print("=" * 65)
print("         📊 SENTIMENT ANALYSIS — FINAL SUMMARY REPORT")
print("=" * 65)
print(f"\n📦 Total Tweets Analyzed : {len(df)}")
print(f"📂 Categories Covered    : {df['category'].nunique()}")

print("\n🔷 TextBlob Results:")
for s, c in df['tb_sentiment'].value_counts().items():
    pct = c / len(df) * 100
    bar = '█' * int(pct / 2)
    print(f"   {s:<12} {bar:<30} {c:3d} tweets ({pct:.1f}%)")

print("\n🔶 VADER Results:")
for s, c in df['vader_sentiment'].value_counts().items():
    pct = c / len(df) * 100
    bar = '█' * int(pct / 2)
    print(f"   {s:<12} {bar:<30} {c:3d} tweets ({pct:.1f}%)")

agreement = (df['tb_sentiment'] == df['vader_sentiment']).mean() * 100
print(f"\n🤝 Model Agreement Rate  : {agreement:.1f}%")
print(f"📉 Avg TextBlob Polarity : {df['tb_polarity'].mean():.3f}")
print(f"📉 Avg VADER Compound    : {df['vader_compound'].mean():.3f}")

print("\n💡 Key Takeaways:")
print("   • VADER outperforms TextBlob on social media text (emojis, slang, caps)")
print("   • Sarcasm detection remains a major challenge for rule-based models")
print("   • TextBlob is better for formal/clean text; VADER for raw tweets")
print("   • For production use, consider fine-tuned transformers (e.g., BERT)")
print("\n✅ Project Complete!")

## 💾 Step 9: Export Results to CSV

In [ ]:
output_cols = ['tweet_id', 'username', 'text', 'category', 'cleaned_text',
               'tb_polarity', 'tb_subjectivity', 'tb_sentiment',
               'vader_compound', 'vader_pos', 'vader_neg', 'vader_neu', 'vader_sentiment',
               'likes', 'retweets']

df[output_cols].to_csv('sentiment_analysis_results.csv', index=False)
print("✅ Results exported to 'sentiment_analysis_results.csv'")
print(f"   Rows: {len(df)} | Columns: {len(output_cols)}")

# Preview
df[output_cols].head()